# Simple Operations on Periodic Splines
The data samples at the integers are represented with circles and stem lines. The sample at the origin, as well as its periodized replicates, is indicated by a red circle and stem line; the domain of the curves encompasses two periods. The thin blue curve is the result of the operation applied to the thick gray curve.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 8 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay
max_param = 2.0 # Maximal absolute parameter

# Initial random periodic cubic spline
s0 = sk.PeriodicSpline1D.from_spline_coeff(np.random.standard_normal(6), degree = 3)

# Parameter widget
parameter_widget = widgets.FloatSlider(
    value = 0.8,
    min = -max_param,
    max = max_param,
    description = "constant"
)

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    param = 0.8,
    op = 0
):
    global s0

    # Update of the spline
    if s0.period != period:
        s0 = sk.PeriodicSpline1D.from_spline_coeff(
            np.random.standard_normal(period),
            degree = s0.degree
        )
    s0.degree = degree
    s0.delay = delay

    # Selection of the operation
    if 0 == op: # Delayed by a Constant
        s = s0.delayed_by(param)
        parameter_widget.disabled = False
    elif 1 == op: # Plus a Constant
        s = s0.plus(param)
        parameter_widget.disabled = False
    elif 2 == op: # Times a Constant
        s = s0.times(param)
        parameter_widget.disabled = False
    elif 3 == op: # Negated
        s = s0.negated()
        parameter_widget.disabled = True
    elif 4 == op: # Mirrored
        s = s0.mirrored()
        parameter_widget.disabled = True

    # Domain
    plotdomain = sk.interval.Closed((-period - 0.5, period + 0.5))
    # Dynamic range
    image = {s0.image(), s.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))

    # Plot of the splines
    subplot = plt.subplots()
    s0.plot(
        subplot,
        plotdomain = plotdomain,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "#E0E0E0",
        curve_lw = 7.0,
        curve_markerfmt = "oC7",
        curvestem_linefmt = "-C7",
        knot_color = "C7"
    )
    s.plot(
        subplot,
        plotdomain = plotdomain,
        plotrange = plotrange,
        plotpoints = 200 + 1
    )
    plt.show()

widgets.interactive(
    update_plot,
    period = (1, max_period),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    param = parameter_widget,
    op = widgets.RadioButtons(
        options = [
            ("Additional Delay by a Constant", 0),
            ("Plus a Constant", 1),
            ("Times a Constant", 2),
            ("Negated", 3),
            ("Mirrored", 4)
        ],
        value = 0,
        description = "Operation:",
        disabled = False
    )
)
